# Snap2 Cursive OCR Fine-Tuning (PARSeq -> ONNX)

**Zero LLM tokens required.** Run this notebook on a free **Google Colab T4 GPU** (or Kaggle GPU) to fine-tune `baudm/parseq` on your 20,000-sample Snap2 cursive dataset (`training/dataset.zip`), track Validation CER each epoch, save `best_parseq.pt`, verify the exported ONNX graph with `onnxruntime`, and download a drop-in `parseq_cursive.onnx` for `Snap2/public/models/parseq/model.onnx`.

In [ ]:
# 1. Verify GPU and install dependencies
!nvidia-smi
!pip install -q timm pytorch-lightning onnx onnxruntime pillow

In [ ]:
# 2. Clone official PARSeq repo
!git clone --depth 1 https://github.com/baudm/parseq.git
%cd parseq

In [ ]:
# 3. Load dataset.zip (from Google Drive recommended for ~80MB zip, or direct browser upload)
import zipfile
from pathlib import Path

USE_GOOGLE_DRIVE = False  # Set True if you uploaded dataset.zip to MyDrive/Snap2/dataset.zip
ZIP_PATH = '/content/drive/MyDrive/Snap2/dataset.zip'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    from google.colab import files
    uploaded = files.upload()
    ZIP_PATH = next(iter(uploaded.keys()))

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content/snap2_dataset')
print('Extracted dataset to /content/snap2_dataset')

In [ ]:
# 4. Fine-tune pretrained PARSeq with per-epoch Validation CER & best-checkpoint saving
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).to(device)
# Silence Lightning self.log() when running in a plain PyTorch loop (no Trainer attached)
model.log = lambda *args, **kwargs: None

img_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5),
])

def edit_distance(a: str, b: str) -> int:
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev if ca == cb else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

class TsvOcrDataset(Dataset):
    def __init__(self, tsv_path):
        p = Path(tsv_path)
        self.base = p.parent
        self.items = []
        for i, line in enumerate(p.read_text(encoding='utf-8').splitlines()):
            if i == 0 and line.startswith('rel_path\t'):
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                ip = (self.base / parts[0]).resolve()
                if ip.is_file():
                    self.items.append((ip, parts[1].strip()))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        ip, label = self.items[idx]
        img = Image.open(ip).convert('RGB')
        return img_transform(img), label

train_ds = TsvOcrDataset('/content/snap2_dataset/splits/train.tsv')
val_ds = TsvOcrDataset('/content/snap2_dataset/splits/val.tsv')
print(f'Loaded train={len(train_ds)}, val={len(val_ds)}')
assert len(train_ds) > 0 and len(val_ds) > 0, 'Dataset empty — check /content/snap2_dataset extraction!'

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
best_cer = float('inf')
best_ckpt = '/content/best_parseq.pt'

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.training_step((imgs, list(labels)), 0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 20.0)
        optimizer.step()
        total_loss += float(loss.item())
    scheduler.step()

    # Validation CER check
    model.eval()
    tot_edits, tot_chars = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            logits = model(imgs.to(device))
            preds, _ = model.tokenizer.decode(logits.softmax(-1))
            for pred, gt in zip(preds, labels):
                tot_edits += edit_distance(pred.lower(), gt.lower())
                tot_chars += max(1, len(gt))
    val_cer = tot_edits / max(1, tot_chars)
    print(f'Epoch {epoch:02d}/{epochs} | Loss: {total_loss / max(1, len(train_loader)):.4f} | Val CER: {val_cer*100:.2f}%')
    if val_cer < best_cer:
        best_cer = val_cer
        torch.save(model.state_dict(), best_ckpt)
        print(f'  -> Saved new best checkpoint ({best_cer*100:.2f}% CER)')

In [ ]:
# 5. Export best checkpoint to ONNX, verify with onnxruntime, and download
import onnxruntime as ort
import numpy as np
from google.colab import files

model.load_state_dict(torch.load(best_ckpt, map_location='cpu'))
model.eval().cpu()
dummy = torch.randn(1, 3, 32, 128)
out_onnx = '/content/parseq_cursive.onnx'
torch.onnx.export(
    model,
    dummy,
    out_onnx,
    input_names=['images'],
    output_names=['logits'],
    dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

# Sanity-check exported ONNX graph on 5 validation samples
sess = ort.InferenceSession(out_onnx, providers=['CPUExecutionProvider'])
for i in range(min(5, len(val_ds))):
    x, gt = val_ds[i]
    logits = sess.run(None, {'images': x.unsqueeze(0).numpy()})[0]
    pred, _ = model.tokenizer.decode(torch.from_numpy(logits).softmax(-1))
    print(f'ONNX check [{i}]: gt={gt!r} -> pred={pred[0]!r}')

files.download(out_onnx)